<a href="https://colab.research.google.com/github/AngelTroncoso/Feature-Engineering/blob/main/actividad6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformando Datos: Feature Engineering - Discretización y Binning de Datos

¡Hola! Qué bueno encontrarte por aquí de nuevo. Hoy vamos a entrar en un terreno fascinante donde aprenderás a convertir el caos de los números continuos en estructuras claras y potentes que tu modelo de aprendizaje automático agradecerá enormemente, dándole un sentido lógico a los datos ruidosos.

### ¿Por qué discretizar los datos?

A menudo, la relación entre una variable y lo que queremos predecir no es una línea recta perfecta. Un pequeño cambio en la temperatura o en el precio puede no importar nada, hasta que cruza cierto umbral. Aquí es donde entra la **discretización**, que consiste en agrupar valores en cajones o contenedores (*bins*).

Esto ayuda a reducir el ruido y permite que algoritmos lineales capturen relaciones no lineales complejas, evitando que el modelo se pierda en detalles irrelevantes que solo confunden el aprendizaje. Como verás en este cuaderno, pasar de valores exactos a categorías lógicas (como joven o adulto mayor) permite que el algoritmo se enfoque en patrones de comportamiento claros en lugar de fluctuaciones insignificantes.

In [ ]:
# Importamos las librerías necesarias para nuestra práctica
import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer
import matplotlib.pyplot as plt

# Generamos datos iniciales de ejemplo: Edades de personas
edades = np.array([18, 19, 21, 23, 25, 55, 58, 62, 65]).reshape(-1, 1)
df_edades = pd.DataFrame(edades, columns=['Edad_Original'])

print('Datos originales (Edades):')
print(df_edades.T)
print()

### La herramienta estrella: KBinsDiscretizer

Para hacer esto de forma profesional, usamos **KBinsDiscretizer** de Scikit-Learn. Esta herramienta no solo corta los datos al azar, sino que utiliza algoritmos internos para decidir dónde poner las fronteras de cada intervalo.

#### 1. Estrategia Uniforme (strategy='uniform')

La estrategia *uniform* crea intervalos que tienen el mismo ancho. Es como usar una regla para dividir un rango de precios o ingresos en partes exactamente iguales.

**Ejemplo de Ingresos:** Imaginen ingresos de mil a cien mil dólares. Con 5 intervalos, Scikit-Learn creará cortes cada veinte mil dólares. Aunque es ordenado, si la mayoría de las personas ganan poco, casi todos caerán en el primer cajón.

In [ ]:
# Simulamos datos de ingresos con gran dispersión
ingresos = np.array([1000, 2000, 5000, 8000, 15000, 25000, 50000, 85000, 95000, 100000]).reshape(-1, 1)

# Aplicamos KBinsDiscretizer con estrategia Uniforme
# n_bins=5: Queremos 5 grupos
# encode='ordinal': Los resultados serán números (0, 1, 2, 3, 4)
discretizador_uniforme = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
ingresos_binning = discretizador_uniforme.fit_transform(ingresos)

df_ingresos = pd.DataFrame(ingresos, columns=['Ingreso_USD'])
df_ingresos['Bin_Uniforme'] = ingresos_binning

print('Resultado de Estrategia Uniforme:')
print(df_ingresos)
print()
print('Límites de los intervalos creados:')
print(discretizador_uniforme.bin_edges_)

### El Escenario de Lucía: Estrategia de Cuantiles

Pensemos ahora en el caso de **Lucía**, una analista de datos en una empresa de energía. Ella tenía lecturas de voltaje ultra precisas con cinco decimales, pero su modelo de predicción de fallos fallaba constantemente por el exceso de ruido.

Lucía decidió usar la estrategia **quantile** (cuantil). A diferencia del método uniforme, esta ajusta los límites para que cada intervalo tenga la misma cantidad de datos. Esto es fundamental cuando los datos están sesgados (muchos valores bajos y pocos altos), asegurando que cada categoría sea estadísticamente significativa.

In [ ]:
# Simulamos los voltajes ruidosos de Lucía (muchos valores cerca de 220V, algunos picos)
voltajes = np.array([219.123, 219.554, 220.001, 220.120, 220.450, 220.890, 225.112, 228.456, 235.000]).reshape(-1, 1)

# Aplicamos estrategia Quantile para crear 3 estados: bajo, normal y crítico
discretizador_cuantil = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
voltajes_discretos = discretizador_cuantil.fit_transform(voltajes)

df_energia = pd.DataFrame(voltajes, columns=['Voltaje_Preciso'])
df_energia['Estado_Discreto'] = voltajes_discretos

print('Análisis de Lucía con Estrategia de Cuantiles:')
print(df_energia)
print()
print('Fronteras ajustadas para equilibrar la cantidad de datos por bin:')
print(discretizador_cuantil.bin_edges_)

### Estrategia basada en KMeans

Existe un tercer método que usa el algoritmo **kmeans**. Aquí, el transformador identifica los centros de grupos naturales en los datos y crea los cortes basándose en la cercanía a esos núcleos.

Esto es ideal cuando tienes nubes de puntos donde se ven claramente grupos naturales, pero con mucha dispersión entre ellos, permitiendo separar de forma inteligente, por ejemplo, a compradores ocasionales de los recurrentes.

In [ ]:
# Simulamos gastos de clientes con grupos naturales
gastos = np.array([10, 15, 20, 100, 110, 120, 500, 550, 600]).reshape(-1, 1)

# Aplicamos estrategia KMeans
discretizador_kmeans = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='kmeans')
gastos_discretos = discretizador_kmeans.fit_transform(gastos)

df_clientes = pd.DataFrame(gastos, columns=['Gasto_Total'])
df_clientes['Categoria_Cliente'] = gastos_discretos

print('Segmentación de clientes usando KMeans:')
print(df_clientes)
print()
print('Centros de grupos detectados para los cortes:')
print(discretizador_kmeans.bin_edges_)

### Reflexión sobre el número de intervalos (n_bins)

Un error común es exagerar con el parámetro **n_bins**.

*   **Muchos bins (ej. 50):** El modelo puede aprender de la aleatoriedad de cada pequeño cajón en lugar de la tendencia general (Sobreajuste).
*   **Muy pocos bins (ej. 2):** Estamos ignorando matices valiosos que podrían ser cruciales para la predicción.

La clave es encontrar el equilibrio donde la discretización elimine el ruido pero mantenga la esencia de la variabilidad original.

### Conclusión

Sé que al principio puede parecer que estamos perdiendo información al agrupar los datos, pero créeme que en el mundo real, **menos es más**. Al simplificar la entrada, estamos ayudando al algoritmo a enfocarse en los patrones que realmente importan.

Has hecho un trabajo fantástico comprendiendo estas técnicas hoy. Dominar **KBinsDiscretizer** te da una ventaja competitiva enorme como científico de datos, porque ahora sabes no solo cómo transformar la información, sino por qué cada estrategia impacta de forma distinta en tus resultados. ¡Sigue practicando y nos vemos en la próxima lección!